# nnU-Net BraTS 2024 MEN-RT — Full Pipeline

**Before running this notebook:**
1. GPU must be ON → Notebook Settings → Accelerator → **GPU T4 x2** or **P100**
2. Add your BraTS dataset (the two zip files) as a Kaggle Dataset input
3. Add your GitHub token as a Kaggle Secret named `GITHUB_TOKEN`
   - Notebook → Add-ons → Secrets → Add New Secret

**Training plan:**
- This notebook trains **Fold 0 only** for **50 epochs** as a test run
- If it works, run remaining folds in a new session using the last cell

**Expected runtime:** ~3–4 hours on T4 GPU for 50 epochs of 3d_fullres

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, os

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Check available input data ───────────────────────────────────────
# Run this to find the correct path to your BraTS zip files
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# ── Cell 3: Clone repository from GitHub ─────────────────────────────────────
# Reads GITHUB_TOKEN from Kaggle Secrets (Add-ons → Secrets)
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
token   = secrets.get_secret('GITHUB_TOKEN')

repo_url = f'https://{token}@github.com/maryampervaiz-alt/nnunet-training.git'
os.system(f'git clone {repo_url} /kaggle/working/nnunet-training')

print('\nRepository contents:')
os.system('ls /kaggle/working/nnunet-training')

In [ ]:
# ── Cell 4: Install dependencies ─────────────────────────────────────────────
# surface-distance must be installed from GitHub (PyPI version is a different package)
!pip install git+https://github.com/google-deepmind/surface-distance.git -q
!pip install nnunetv2 -q
!pip install nibabel SimpleITK scipy scikit-image pandas matplotlib loguru python-dotenv tqdm rich mlflow -q

print('\nAll packages installed.')

In [ ]:
# ── Cell 5: Configure environment variables ───────────────────────────────────
#
# !! EDIT THE TWO PATHS BELOW !!
# Check Cell 2 output to find the correct paths to your zip files.
# Example: /kaggle/input/brats-men-rt/BraTS2024-MEN-RT-TrainingData.zip
#
TRAIN_ZIP = '/kaggle/input/YOUR-DATASET-NAME/BraTS2024-MEN-RT-TrainingData.zip'
VAL_ZIP   = '/kaggle/input/YOUR-DATASET-NAME/BraTS2024-MEN-RT-ValidationData.zip'

# ── Project paths ─────────────────────────────────────────────────────────────
PROJECT = '/kaggle/working/nnunet-training'

os.environ['nnUNet_raw']          = f'{PROJECT}/nnunet_raw'
os.environ['nnUNet_preprocessed'] = f'{PROJECT}/nnunet_preprocessed'
os.environ['nnUNet_results']      = f'{PROJECT}/checkpoints'
os.environ['DATASET_ID']          = '001'
os.environ['DATASET_NAME']        = 'BraTSMENRT'
os.environ['NNUNET_CONFIGURATION']= '3d_fullres'
os.environ['NNUNET_SEED']         = '42'
os.environ['ES_PATIENCE']         = '50'
os.environ['ES_MIN_DELTA']        = '0.0001'
os.environ['ES_WARMUP']           = '50'
os.environ['NUM_FOLDS']           = '5'
os.environ['CUDA_VISIBLE_DEVICES']= '0'
os.environ['TRAIN_SOURCE']        = TRAIN_ZIP
os.environ['VAL_SOURCE']          = VAL_ZIP
os.environ['LABEL_SUFFIX']        = 'gtv'
os.environ['EXPERIMENT_NAME']     = 'BraTSMENRT_baseline'
os.environ['MLFLOW_TRACKING_URI'] = f'{PROJECT}/experiments/mlruns'

# Create output directories
for d in ['nnunet_raw','nnunet_preprocessed','checkpoints',
          'metrics','results','visualizations','inference_outputs','logs','experiments']:
    os.makedirs(f'{PROJECT}/{d}', exist_ok=True)

print('Environment configured:')
print(f'  nnUNet_raw          : {os.environ["nnUNet_raw"]}')
print(f'  nnUNet_preprocessed : {os.environ["nnUNet_preprocessed"]}')
print(f'  nnUNet_results      : {os.environ["nnUNet_results"]}')
print(f'  TRAIN_SOURCE        : {TRAIN_ZIP}')
print(f'  VAL_SOURCE          : {VAL_ZIP}')

In [ ]:
# ── Cell 6: Change working directory to the repo ──────────────────────────────
# All ! commands after this run inside the repository folder
%cd /kaggle/working/nnunet-training

In [ ]:
# ── Cell 7: STEP 1 — Convert raw BraTS data to nnU-Net format ────────────────
# Reads zip files, renames files, writes dataset.json
# Expected time: 10–20 minutes (extracting + copying ~500 cases)
!python scripts/01_prepare_dataset.py \
    --train "{TRAIN_ZIP}" \
    --val   "{VAL_ZIP}"   \
    --log-dir logs

In [ ]:
# Verify dataset was created correctly
!ls -lh $nnUNet_raw/Dataset001_BraTSMENRT/
!echo '--- imagesTr count ---'
!ls $nnUNet_raw/Dataset001_BraTSMENRT/imagesTr/ | wc -l
!echo '--- labelsTr count ---'
!ls $nnUNet_raw/Dataset001_BraTSMENRT/labelsTr/ | wc -l
!echo '--- dataset.json ---'
!cat $nnUNet_raw/Dataset001_BraTSMENRT/dataset.json

In [ ]:
# ── Cell 8: STEP 2 — nnU-Net Planning and Preprocessing ──────────────────────
# nnU-Net reads the data and automatically decides:
#   - patch size, batch size, network depth
#   - resampling spacing, normalization
# Expected time: 20–40 minutes
!python scripts/02_preprocess.py --log-dir logs

In [ ]:
# Verify preprocessing output
!ls $nnUNet_preprocessed/Dataset001_BraTSMENRT/
!echo '--- Splits file (5-fold CV) ---'
!python -c "import json; s=json.load(open('$nnUNet_preprocessed/Dataset001_BraTSMENRT/splits_final.json')); print(f'{len(s)} folds | fold_0 train={len(s[0][\"train\"])} val={len(s[0][\"val\"])}')" 2>/dev/null || echo 'splits_final.json not found yet'

In [ ]:
# ── Cell 9: STEP 3 — Train Fold 0 (50 epochs test run) ───────────────────────
#
# This is a TEST RUN with 50 epochs to verify the pipeline works.
# For full training (1000 epochs + early stopping), use the last cell.
#
# Expected time: 2–4 hours on T4 GPU
#
!nnUNetv2_train 001 3d_fullres 0 \
    -tr nnUNetTrainerEarlyStopping \
    -p  nnUNetPlans \
    --num_epochs 50

In [ ]:
# Verify checkpoint was saved
CKPT_DIR = f'/kaggle/working/nnunet-training/checkpoints/Dataset001_BraTSMENRT/nnUNetTrainerEarlyStopping__nnUNetPlans__3d_fullres/fold_0'
print(f'Checkpoint directory: {CKPT_DIR}')
!ls -lh "{CKPT_DIR}" 2>/dev/null || echo 'Checkpoint directory not found — training may have failed'

In [ ]:
# ── Cell 10: STEP 4 — Inference on Fold 0 validation cases ───────────────────
# Predicts GTV masks for the held-out validation cases of fold 0
# Expected time: 10–30 minutes
!python scripts/04_inference.py \
    --cv-mode \
    --cv-output-root inference_outputs/cv \
    --log-dir logs

In [ ]:
# Verify predictions exist
!echo '--- Fold 0 predictions ---'
!ls inference_outputs/cv/fold_0/ | head -10
!echo ''
!echo 'Total predictions:'
!ls inference_outputs/cv/fold_0/*.nii.gz 2>/dev/null | wc -l

In [ ]:
# ── Cell 11: STEP 5 — Evaluate Fold 0 ────────────────────────────────────────
# Computes: DSC, HD95, NSD, Precision, Recall, Specificity, Volume Error
# Expected time: 5–15 minutes
!python scripts/05_evaluate.py \
    --cv-mode \
    --results-dir results \
    --show-rankings \
    --log-dir logs

In [ ]:
# ── Cell 12: STEP 6 — Visualizations ─────────────────────────────────────────
# Generates: overlays, violin plots, fold comparison charts, training curves
# Expected time: 2–5 minutes
!python scripts/06_visualize.py \
    --all \
    --cv-mode \
    --results-dir results \
    --metrics-dir metrics \
    --output-dir visualizations \
    --log-dir logs

In [ ]:
# ── Cell 13: View Results ─────────────────────────────────────────────────────
import pandas as pd

result_files = [
    'results/fold_0_per_case.csv',
    'results/cv_combined.csv',
]

for f in result_files:
    if os.path.exists(f):
        df = pd.read_csv(f)
        cols = ['case_id','dice','hd95','nsd','precision','recall']
        cols = [c for c in cols if c in df.columns]
        print(f'\n=== {f} ({len(df)} cases) ===')
        print(df[cols].round(4).to_string(index=False))
        print(f'\nMean DSC  : {df["dice"].mean():.4f} ± {df["dice"].std():.4f}')
        if 'hd95' in df.columns:
            finite_hd = df['hd95'].replace([float('inf')], float('nan'))
            print(f'Mean HD95 : {finite_hd.mean():.2f} mm')
        if 'nsd' in df.columns:
            print(f'Mean NSD  : {df["nsd"].mean():.4f}')
    else:
        print(f'{f} not found yet')

In [ ]:
# ── Cell 14: Show overlay images ──────────────────────────────────────────────
from IPython.display import Image, display
from pathlib import Path

overlay_dir = Path('visualizations/overlays')
if overlay_dir.exists():
    images = sorted(overlay_dir.glob('*.png'))[:3]  # show first 3
    for img_path in images:
        print(f'\n{img_path.name}')
        display(Image(str(img_path), width=900))
else:
    print('No overlay images found — run Cell 12 first')

In [ ]:
# ── Cell 15: Show metric plots ────────────────────────────────────────────────
for plot in ['visualizations/metrics_violin.png',
             'visualizations/metrics_boxplot.png',
             'visualizations/volume_scatter.png']:
    if os.path.exists(plot):
        print(plot)
        display(Image(plot, width=900))

In [ ]:
# ── Cell 16: Save everything to Kaggle output ─────────────────────────────────
# Kaggle deletes /kaggle/working/nnunet-training when the session ends.
# This cell copies all important outputs to /kaggle/working/ which IS saved.
import shutil

SAVE = '/kaggle/working/outputs'
os.makedirs(SAVE, exist_ok=True)

to_save = [
    ('results',           f'{SAVE}/results'),
    ('metrics',           f'{SAVE}/metrics'),
    ('visualizations',    f'{SAVE}/visualizations'),
    ('inference_outputs', f'{SAVE}/inference_outputs'),
    # Save fold 0 checkpoint (important!)
    ('checkpoints',       f'{SAVE}/checkpoints'),
]

for src, dst in to_save:
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Saved: {src} → {dst}')
    else:
        print(f'Skipped (not found): {src}')

print('\nAll outputs saved to /kaggle/working/outputs/')
!du -sh /kaggle/working/outputs/*

---

## If Fold 0 Passed — Run Remaining Folds

Start a **new Kaggle session**, clone the repo again, run Cells 1–8, then run the cell below **instead of Cell 9**.

This will train folds 1, 2, 3, 4 with full 1000 epochs + early stopping.

In [ ]:
# ── FULL TRAINING — Folds 1–4 (run in new session after fold 0 passed) ────────
# This uses the full pipeline script with early stopping.
# Expected time: 12–24 hours per fold on T4 GPU
#
# Uncomment and run one fold at a time to stay within Kaggle's 12-hour limit.

FOLD = '1'   # change to 2, 3, 4 for subsequent sessions

!nnUNetv2_train 001 3d_fullres {FOLD} \
    -tr nnUNetTrainerEarlyStopping \
    -p  nnUNetPlans

# Then save the checkpoint before session ends:
!cp -r checkpoints /kaggle/working/outputs/checkpoints